In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Set Jupyter renderer (iframe has the best compatibility)
pio.renderers.default = "iframe"
pd.set_option('future.no_silent_downcasting', True)

# ==========================================
# 1. Define file mapping and reading
# ==========================================
file_map = {
    2018: 'hosp-epis-stat-admi-diag-2018-19-tab.xlsx',
    2019: 'hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx',
    2020: 'hosp-epis-stat-admi-diag-2020-21-tab.xlsx',
    2021: 'hosp-epis-stat-admi-diag-2021-22-tab.xlsx'
}

def smart_read(filename, sheet_keywords):
    xl = pd.ExcelFile(filename)
    target_sheet = [s for s in xl.sheet_names if sheet_keywords in s]
    if not target_sheet: return None
    sheet_name = target_sheet[0]
    raw_head = pd.read_excel(xl, sheet_name=sheet_name, header=None, nrows=15)
    mask = raw_head.apply(lambda row: row.astype(str).str.contains('Male|Finished|Female', case=False).any(), axis=1)
    header_row = mask.idxmax() if mask.any() else 10
    df = pd.read_excel(xl, sheet_name=sheet_name, skiprows=header_row)
    df.columns = [str(c).replace('\n', ' ').strip() for c in df.columns]
    df.rename(columns={df.columns[0]: 'Code_Or_Name', df.columns[1]: 'Description'}, inplace=True)
    return df

df_3char_list, df_sum_list = [], []
for year, filename in file_map.items():
    print(f"Reading data for year {year}...")
    d3 = smart_read(filename, '3 Character')
    ds = smart_read(filename, 'Summary')
    if d3 is not None: d3['Year'] = year; df_3char_list.append(d3)
    if ds is not None: ds['Year'] = year; df_sum_list.append(ds)

df_3char_raw = pd.concat(df_3char_list, ignore_index=True)
df_sum_raw = pd.concat(df_sum_list, ignore_index=True)

# ==========================================
# 2. Data cleaning and mapping dictionary preparation
# ==========================================
df_sum = df_sum_raw[~df_sum_raw['Code_Or_Name'].astype(str).str.contains('Total', case=False, na=False)].copy()
df_sum['Disease_Category'] = df_sum['Code_Or_Name'].astype(str).str.extract(r'([A-Z])')[0].str.upper()
df_sum = df_sum.dropna(subset=['Disease_Category']).replace(['*', '-', ' ', '..'], [3, np.nan, np.nan, np.nan]).infer_objects(copy=False)

# Mapping 1: Category code -> Category full description
chapter_map = df_sum.drop_duplicates('Disease_Category').set_index('Disease_Category')['Description'].to_dict()

df_3char = df_3char_raw[~df_3char_raw['Code_Or_Name'].astype(str).str.contains('Total', case=False, na=False)].copy()
df_3char = df_3char.replace(['*', '-', ' ', '..'], [3, np.nan, np.nan, np.nan]).infer_objects(copy=False)
df_3char['Episodes'] = pd.to_numeric(df_3char['Finished consultant episodes'], errors='coerce').fillna(0)
df_3char['Disease_Category'] = df_3char['Code_Or_Name'].astype(str).str[0].str.upper()
df_3char['Diagnosis_Code'] = df_3char['Code_Or_Name'].astype(str).str.strip()

# Mapping 2: 3-character code -> Specific disease description
h_map = df_3char.drop_duplicates('Diagnosis_Code').set_index('Diagnosis_Code')['Description'].to_dict()

# Age group calculation
age_groups = {
    'Age 0-18': ['Age 0', 'Age 1-4', 'Age 5-9', 'Age 10-14', 'Age 15', 'Age 16', 'Age 17', 'Age 18'],
    'Age 19-39': ['Age 19', 'Age 20-24', 'Age 25-29', 'Age 30-34', 'Age 35-39'],
    'Age 40-59': ['Age 40-44', 'Age 45-49', 'Age 50-54', 'Age 55-59'],
    'Age 60+': ['Age 60-64', 'Age 65-69', 'Age 70-74', 'Age 75-79', 'Age 80-84', 'Age 85-89', 'Age 90+']
}
for label, cols in age_groups.items():
    valid_cols = [c for c in cols if c in df_sum.columns]
    df_sum[label] = df_sum[valid_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)

# Generate melted tables
df_gender = df_sum.melt(id_vars=['Year', 'Disease_Category'], value_vars=['Male', 'Female'], var_name='Gender', value_name='v').rename(columns={'v':'Episodes'})
df_age = df_sum.melt(id_vars=['Year', 'Disease_Category'], value_vars=list(age_groups.keys()), var_name='Age_Group', value_name='v').rename(columns={'v':'Episodes'})
df_sum['Episodes'] = df_sum[list(age_groups.keys())].sum(axis=1)

# ==========================================
# 3. Core calculation function
# ==========================================
def create_nodes(df, group_cols, id_prefix, parent_val, weight=1, is_leaf_3char=False):
    yearly = df.groupby(['Year'] + group_cols)['Episodes'].sum().reset_index()
    v18 = yearly[yearly['Year'].isin([2018, 2019])].groupby(group_cols)['Episodes'].mean().reset_index(name='v18')
    v20 = yearly[yearly['Year'].isin([2020, 2021])].groupby(group_cols)['Episodes'].mean().reset_index(name='v20')
    m = pd.merge(v18, v20, on=group_cols, how='outer').fillna(0)
    m['rate'] = np.where(m['v18'] == 0, 0, (m['v20'] - m['v18']) / m['v18'])
    
    if len(group_cols) > 1:
        m['id'] = id_prefix + m[group_cols[0]].astype(str) + "_" + m[group_cols[1]].astype(str)
        # Fix: Distinguish between bottom leaf nodes and drill-down category nodes for naming and hovering
        if is_leaf_3char:
            m['name'] = m[group_cols[0]].astype(str)
            m['hover'] = m['name'].map(h_map)
        else:
            m['name'] = m[group_cols[1]].astype(str)
            m['hover'] = m['name'].map(chapter_map)
    else:
        m['id'] = id_prefix + m[group_cols[0]].astype(str)
        m['name'] = m[group_cols[0]].astype(str)
        m['hover'] = m['name'].map(chapter_map) if id_prefix == 'DC_' else ""
    
    m['parent'] = parent_val(m) if callable(parent_val) else parent_val
    m['weight'] = m['v20'] * weight
    return m

# ==========================================
# 4. Construct node tree and ultimate filter interceptor
# ==========================================
df_root = pd.DataFrame([
    {'id': 'R', 'name': 'In-depth Analysis of NHS Admission Trends', 'parent': '', 'v18': 0, 'v20': 0, 'rate': 0, 'weight': 0, 'hover': ''},
    {'id': 'D', 'name': 'By Disease Category', 'parent': 'R', 'v18': 0, 'v20': 0, 'rate': 0, 'weight': 0, 'hover': ''},
    {'id': 'G', 'name': 'By Gender', 'parent': 'R', 'v18': 0, 'v20': 0, 'rate': 0, 'weight': 0, 'hover': ''},
    {'id': 'A', 'name': 'By Age Group', 'parent': 'R', 'v18': 0, 'v20': 0, 'rate': 0, 'weight': 0, 'hover': ''}
])

n_dis_cat = create_nodes(df_sum, ['Disease_Category'], 'DC_', 'D', weight=0)
n_dis_leaf = create_nodes(df_3char, ['Diagnosis_Code', 'Disease_Category'], 'DL_', lambda x: 'DC_'+x['Disease_Category'], weight=8, is_leaf_3char=True)
n_gen_type = create_nodes(df_gender, ['Gender'], 'GT_', 'G', weight=0)
n_gen_cat = create_nodes(df_gender, ['Gender', 'Disease_Category'], 'GC_', lambda x: 'GT_'+x['Gender'], weight=1)
n_age_type = create_nodes(df_age, ['Age_Group'], 'AT_', 'A', weight=0)
n_age_cat = create_nodes(df_age, ['Age_Group', 'Disease_Category'], 'AC_', lambda x: 'AT_'+x['Age_Group'], weight=1)

df_final = pd.concat([df_root, n_dis_cat, n_dis_leaf, n_gen_type, n_gen_cat, n_age_type, n_age_cat], ignore_index=True)
df_final['hover'] = df_final['hover'].fillna("No detailed description")

# 🌟 Core filter to fix the blank screen issue 🌟
# 1. Remove leaf nodes with a weight of 0 (keep the branch structure)
is_structure = df_final['id'].isin(['R', 'D', 'G', 'A']) | df_final['id'].str.startswith(('DC_', 'GT_', 'AT_'))
has_data = df_final['weight'] > 0
df_final = df_final[is_structure | has_data].copy()

# 2. Remove "orphan nodes" that cannot find their parent nodes (this is the culprit causing Plotly's blank screen)
valid_ids = set(df_final['id'])
df_final = df_final[df_final['parent'].isin(valid_ids) | (df_final['parent'] == '')]

# ==========================================
# 5. Plotting
# ==========================================
print(f"Data processing complete. Valid plotting nodes: {len(df_final)}")

fig = px.treemap(
    df_final, ids='id', names='name', parents='parent', values='weight',
    color='rate', color_continuous_scale='BrBG_r', color_continuous_midpoint=0,
    range_color=[-0.4, 0.4], title="NHS Admission Trends",
    custom_data=['hover', 'v18', 'v20', 'rate']
)

fig.update_traces(
    hovertemplate="""
    <b>Name: %{label}</b><br>
    Description: %{customdata[0]}<br>
    <br>
    18-19 Average: %{customdata[1]:,.0f}<br>
    20-21 Average: %{customdata[2]:,.0f}<br>
    <b>Growth Rate: %{customdata[3]:.2%}</b>
    <extra></extra>
    """
)

fig.update_layout(margin=dict(t=50, l=10, r=10, b=10), width=1100, height=750)
fig.show()

Reading data for year 2018...
Reading data for year 2019...
Reading data for year 2020...
Reading data for year 2021...
Data processing complete. Valid plotting nodes: 1782
